In [ ]:
%pip install -r ../requirements.txt

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score

In [2]:
feat = pd.read_csv('../data/feature_table.tsv', sep='\t', index_col=0)
feat.columns = feat.columns.str.replace("_merged", "", regex=False)
cat = pd.read_csv('../data/samples_categories.tsv', sep='\t', header=None, names=['sample', 'category'])
cat["sample"] = cat["sample"].str.replace("_merged", "", regex=False)

In [3]:
X = feat.T 
y = cat.set_index('sample').loc[X.index, 'category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

param_grid = {'n_estimators': [200, 500, 800],
              'max_depth': [5, 10, None],
              'min_samples_leaf': [1, 2, 3]}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    param_grid, cv=cv, n_jobs=-1)

grid.fit(X_train, y_train)

best_model_rf = grid.best_estimator_

y_pred = best_model_rf.predict(X_test)

In [4]:
print('Best params', grid.best_params_)
print("Classification report")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Best params {'max_depth': 5, 'min_samples_leaf': 1, 'n_estimators': 200}
Classification report
              precision    recall  f1-score   support

     disease       1.00      1.00      1.00         9
      health       1.00      1.00      1.00         9

    accuracy                           1.00        18
   macro avg       1.00      1.00      1.00        18
weighted avg       1.00      1.00      1.00        18

[[9 0]
 [0 9]]


In [22]:
mdi = pd.Series(best_model_rf.feature_importances_, index=X_train.columns)

Extracting important features based on contigues

In [31]:
from Bio import SeqIO
import pandas as pd

def get_lengths(fasta_path, prefix):
    lengths = {}

    for record in SeqIO.parse(fasta_path, "fasta"):
        feature_num = record.id.split("_")[0]
        feature_name = f"{prefix}_{feature_num}"
        lengths[feature_name] = max(lengths.get(feature_name, 0), len(record.seq))

    return lengths

health_lengths = get_lengths("../data/components.seq_he.fasta", "health")
disease_lengths = get_lengths("../data/components.seq.fasta", "disease")

lengths = {**health_lengths, **disease_lengths}

lengths_df = pd.DataFrame.from_dict(
    lengths, orient="index", columns=["length"]
)

imp_df = mdi.reset_index()
imp_df.columns = ["feature", "importance"]

merged = imp_df.merge(
    lengths_df,
    left_on="feature",
    right_index=True,
    how="left"
)

In [32]:
filtered = merged[merged["length"] >= 100]
top_long = filtered.sort_values("importance", ascending=False).head(15)
print(top_long)

          feature  importance  length
13       health_2    0.010000    1532
43      health_38    0.006250     423
59   disease_1349    0.005000     104
95       health_0    0.005000    2274
111   disease_702    0.004718     102
133  disease_1112    0.004179     114
135   disease_761    0.003977     102
136   disease_356    0.003958     145
137   disease_990    0.003939     158
147   disease_255    0.003459     132
162  disease_1741    0.000788     140
183    health_211    0.000280     102
191    health_503    0.000278     101
303  disease_5086    0.000000     112
391  disease_5279    0.000000     103


Validation on other samples

In [5]:
fval = pd.read_csv('../data/feature_table_val.tsv', sep='\t', index_col=0)

In [6]:
X_val = fval.T

In [7]:
y_pred_val = best_model_rf.predict(X_val)

In [8]:
y_pred_val

array(['disease', 'disease', 'disease', 'disease', 'disease', 'disease',
       'disease', 'disease', 'disease', 'disease', 'disease', 'disease',
       'disease', 'disease', 'disease', 'disease', 'disease', 'disease',
       'disease', 'disease', 'disease', 'disease', 'disease', 'disease',
       'disease', 'disease', 'disease', 'disease', 'disease', 'disease',
       'disease', 'disease', 'disease', 'disease', 'disease', 'disease',
       'disease', 'disease', 'disease', 'disease', 'disease', 'disease',
       'disease', 'disease', 'disease', 'disease', 'disease', 'disease',
       'disease', 'disease', 'disease', 'disease', 'disease', 'disease',
       'disease', 'disease', 'disease', 'disease', 'disease', 'disease',
       'disease', 'disease', 'disease', 'disease', 'disease', 'disease',
       'disease', 'disease', 'disease', 'disease', 'disease', 'disease',
       'disease', 'disease', 'disease', 'disease', 'disease', 'disease',
       'disease', 'disease', 'disease', 'disease', 

Validation on self samples with split (as in metafx)

In [9]:
df_il = pd.read_csv('../data/kraken_il_species_abundance.tsv', sep='\t')
df_he = pd.read_csv('../data/kraken_he_species_abundance.tsv', sep='\t')
df_he = df_he.set_index("taxon")
df_il = df_il.set_index('taxon')
df_all = pd.concat([df_il, df_he], axis=1).fillna(0)

In [10]:
test_samples = ['ERR2784729', 'ERR2784780', 'SRR11047641', 'SRR11047647', 'ERR2784742', 
                'ERR2784783', 'SRR11047642', 'SRR11047648', 'ERR2784754', 'ERR2784798', 
                'SRR11047643', 'SRR11047649', 'ERR2784757', 'ERR2784801', 'SRR11047644', 
                'SRR11047653', 'ERR2784761', 'ERR2784804', 'SRR11047645', 'SRR11047673']

df_test = df_all[test_samples]

In [11]:
df_train = df_all.drop(columns=test_samples)

In [12]:
meta_il = pd.DataFrame({'sample': df_il.columns, 'group': 'disease'})
meta_he = pd.DataFrame({'sample': df_he.columns, 'group': 'healthy'})
meta = pd.concat([meta_il, meta_he], ignore_index=True)
meta = meta.set_index('sample')

In [13]:
df_train_rel = df_train.div(df_train.sum(axis=0), axis=1)
df_test_rel = df_test.div(df_test.sum(axis=0), axis=1)
df_train_rel_t = df_train_rel.T
df_test_rel_t = df_test_rel.T

In [14]:
X_train = df_train_rel.T
X_test = df_test_rel.T

y_train = meta.loc[X_train.index, "group"]
y_test = meta.loc[X_test.index, "group"] 

In [ ]:
# param_grid = {
#     "n_estimators": [300, 500, 1000],
#     "max_depth": [None, 3, 5, 10],
#     "min_samples_leaf": [1, 2, 5, 10],
#     "min_samples_split": [2, 5, 10, 20],
#     "max_features": ["sqrt", "log2", 0.1, 0.3],
#     "class_weight": [None, "balanced", "balanced_subsample"],
# }

param_grid = {
    "n_estimators": [300],
    "max_depth": [None],
    "min_samples_leaf": [1, 2],
    "min_samples_split": [2, 5],
    "max_features": ["sqrt"],
    "class_weight": [None, "balanced"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

gs = GridSearchCV(
    rf,
    param_grid=param_grid,
    scoring="balanced_accuracy",
    cv=cv,
    n_jobs=-1,
    verbose = 2,
    refit=True
)

gs.fit(X_train, y_train)

model = gs.best_estimator_
y_pred = model.predict(X_test)


In [45]:
print("Best params:", gs.best_params_)
print("Predicted classes:")
print(pd.Series(y_pred, index=X_test.index).value_counts())

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("balanced accuracy:", balanced_accuracy_score(y_test, y_pred))

Best params: {'class_weight': None, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Predicted classes:
healthy    10
disease    10
Name: count, dtype: int64
[[10  0]
 [ 0 10]]
              precision    recall  f1-score   support

     disease       1.00      1.00      1.00        10
     healthy       1.00      1.00      1.00        10

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20

balanced accuracy: 1.0
